# AICTE | IBM SkillsBuild Data Analytics with AI Internship 2026 | BharatCares
## Customer Churn Retention & Revenue Optimization Platform
**Candidate Name:** Akshad Viresh Makhana  
**Internship ID:** IBMUEDA4101  
**Institution:** Sanjivani University, Kopargaon, Maharashtra  
**Degree:** TY B.Tech CSE (AI & DS)  
**Live Application:** [https://akshad-bharatcares-retention.streamlit.app/](https://akshad-bharatcares-retention.streamlit.app/)  
**GitHub Repository:** [https://github.com/akshad1007/bharatcares-telco-churn-retention](https://github.com/akshad1007/bharatcares-telco-churn-retention)  

---
### Analytical Workflow Paradigm:
`Data -> Information -> Insight -> Decision -> Action`


In [ ]:
# Step 1: Environment Setup and Library Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

print("All analytics and machine learning libraries successfully imported.")


In [ ]:
# Step 2: Data Ingestion & Schema Inspection
dataset_path = "Telco-Customer-Churn.csv"
df = pd.read_csv(dataset_path)

print(f"Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()


In [ ]:
# Step 3: Data Wrangling & Cleaning Pipeline
# 1. TotalCharges contains whitespace blanks for 11 tenure=0 subscribers -> coerce to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

# 2. Recode SeniorCitizen for human interpretability
df['SeniorCitizen'] = df['SeniorCitizen'].map({1: 'Yes', 0: 'No'})

# 3. Clean binary target
df['Churn_Numeric'] = (df['Churn'] == 'Yes').astype(int)

print(f"Missing values remaining in dataset: {df.isnull().sum().sum()}")
print(f"Baseline Churn Rate: {df['Churn_Numeric'].mean()*100:.2f}% ({df['Churn_Numeric'].sum()} churned out of {len(df)})")


In [ ]:
# Step 4: Exploratory Data Analysis (EDA) - Contract & Payment Attrition
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Churn rate by Contract
contract_churn = df.groupby('Contract')['Churn_Numeric'].mean() * 100
axes[0].bar(contract_churn.index, contract_churn.values, color=['#e74c3c', '#3498db', '#2ecc71'])
axes[0].set_title("Churn Rate (%) by Contract Type", fontsize=12, fontweight='bold')
axes[0].set_ylabel("Churn Percentage (%)")
for i, v in enumerate(contract_churn.values):
    axes[0].text(i, v + 1, f"{v:.1f}%", ha='center', fontweight='bold')

# 2. Churn rate by Payment Method
payment_churn = df.groupby('PaymentMethod')['Churn_Numeric'].mean() * 100
axes[1].barh(payment_churn.index, payment_churn.values, color=['#e67e22', '#9b59b6', '#34495e', '#1abc9c'])
axes[1].set_title("Churn Rate (%) by Payment Method", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Churn Percentage (%)")
for i, v in enumerate(payment_churn.values):
    axes[1].text(v + 1, i, f"{v:.1f}%", va='center', fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
# Step 5: Tenure Cohort & Monthly Charges Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Tenure distribution by Churn
sns.kdeplot(df[df['Churn'] == 'No']['tenure'], ax=axes[0], label='Retained (No)', shade=True, color='#2ecc71')
sns.kdeplot(df[df['Churn'] == 'Yes']['tenure'], ax=axes[0], label='Churned (Yes)', shade=True, color='#e74c3c')
axes[0].set_title("Tenure (Months) Distribution by Churn", fontsize=12, fontweight='bold')
axes[0].legend()

# Monthly Charges by Churn
sns.boxplot(x='Churn', y='MonthlyCharges', data=df, ax=axes[1], palette=['#2ecc71', '#e74c3c'])
axes[1].set_title("Monthly Charges ($) Distribution by Churn", fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
# Step 6: Feature Engineering & One-Hot Encoding
features_to_drop = ['customerID', 'Churn', 'Churn_Numeric']
X_raw = df.drop(columns=features_to_drop)
y = df['Churn_Numeric']

# One-hot encode categoricals
X = pd.get_dummies(X_raw, drop_first=True)

# Train/Test Split (80/20 Stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"Testing set:  {X_test.shape[0]} samples, {X_test.shape[1]} features")


In [ ]:
# Step 7: Machine Learning Model Benchmarking
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest Classifier": RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42),
    "Gradient Boosting Classifier": HistGradientBoostingClassifier(max_iter=100, random_state=42)
}

results = []

for name, model in models.items():
    if name == "Logistic Regression":
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]
        
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    
    results.append({
        "Model": name,
        "Accuracy": f"{acc*100:.2f}%",
        "Precision": f"{prec*100:.2f}%",
        "Recall": f"{rec*100:.2f}%",
        "F1-Score": f"{f1:.4f}",
        "ROC-AUC": f"{auc:.4f}"
    })

results_df = pd.DataFrame(results)
print("=== MACHINE LEARNING MODEL BENCHMARK RESULTS ===")
results_df


In [ ]:
# Step 8: Feature Importance Extraction (Random Forest)
rf_model = models["Random Forest Classifier"]
importances = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
importances.head(10).plot(kind='barh', color='#2980b9')
plt.title("Top 10 Feature Drivers of Customer Churn (Random Forest)", fontsize=13, fontweight='bold')
plt.xlabel("Gini Feature Importance")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
# Step 9: Decision & Action - Enterprise Financial ROI Calculator
total_churned_customers = int(df['Churn_Numeric'].sum())
avg_monthly_charges = float(df[df['Churn_Numeric'] == 1]['MonthlyCharges'].mean())
annual_revenue_at_risk = total_churned_customers * avg_monthly_charges * 12

# Retention Campaign Parameters
intervention_success_rate = 0.15   # 15% of at-risk customers successfully retained
incentive_cost_per_cust = 45.00    # $45 credit incentive
campaign_overhead_cost = 25000.00  # $25,000 campaign operational overhead

retained_customers = int(total_churned_customers * intervention_success_rate)
annual_revenue_saved = retained_customers * avg_monthly_charges * 12
total_campaign_cost = (retained_customers * incentive_cost_per_cust) + campaign_overhead_cost
net_annual_benefit = annual_revenue_saved - total_campaign_cost
roi_percentage = (net_annual_benefit / total_campaign_cost) * 100

print("=== BHARATCARES STRATEGIC RETENTION ROI ANALYSIS ===")
print(f"Total Customers at Churn Risk:     {total_churned_customers:,}")
print(f"Annual Gross Revenue at Risk:      ${annual_revenue_at_risk:,.2f}")
print(f"Customers Successfully Retained:   {retained_customers:,}")
print(f"Gross Annual Revenue Preserved:    ${annual_revenue_saved:,.2f}")
print(f"Total Retention Investment Cost:   ${total_campaign_cost:,.2f}")
print(f"Net Annual Preserved Benefit:      ${net_annual_benefit:,.2f}")
print(f"Projected Return on Investment:    {roi_percentage:.1f}%")
